In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data, load_test_data
from models.vanilla_rnn import VanillaRNN
from sklearn.metrics import mean_absolute_error

 # Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
TEST_PATH = PROJECT_ROOT / "datasets_challenge" / "test"
DIST_CSV = PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv"
MODEL_DIR = PROJECT_ROOT / "expert_models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

Project root set to: c:\mait_g2_drift_challenge


In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = VanillaRNN(
    input_size=22,          # number of features
    hidden_size=48,
    num_layers=1,
    dropout=0.0
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [21]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [22]:
def validate(model, dataloader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return mean_absolute_error(targets, preds)


In [23]:
print("Loading train data...")
train_loader, val_loader, pipeline = load_data(
            train_path=TRAIN_PATH,
            dist_csv_path=DIST_CSV,
            val_file_name="train_part_017.csv", 
            columns_to_drop=["sideSlip", "vyCG", "time", "longitude","latitude"],
            sequence_length=20,
            threshold=True,
            threshold_value=3.0,
            threshold_column="vxCG",
            target='sideSlip',
            batchsize=64
        )

Loading train data...
VALIDATION: Loading train_part_017.csv
Calculating weights for WeightedRandomSampler...
Max weight used: 7.0710678118654755
count    136559.000000
mean          1.309014
std           1.070870
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           7.071068
dtype: float64


In [24]:
EPOCHS = 20
best_val_mae = float("inf")
best_model_state = None

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,   # ← weighted sampler is used automatically
        optimizer,
        criterion,
        device 
    )

    val_mae = validate(
        model,
        val_loader,     # ← no sampler, true distribution
        device
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )

    # Save best model (important for snapshot / ensemble)
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_model_state = model.state_dict()
model.load_state_dict(best_model_state)
print(f"✓ Best validation MAE: {best_val_mae:.4f}")   

Epoch 01 | Train Loss: 0.3882 | Val MAE: 0.3575
Epoch 02 | Train Loss: 0.2449 | Val MAE: 0.4396
Epoch 03 | Train Loss: 0.2211 | Val MAE: 0.3762
Epoch 04 | Train Loss: 0.2077 | Val MAE: 0.4223
Epoch 05 | Train Loss: 0.2015 | Val MAE: 0.3323
Epoch 06 | Train Loss: 0.1955 | Val MAE: 0.3186
Epoch 07 | Train Loss: 0.1904 | Val MAE: 0.3164
Epoch 08 | Train Loss: 0.1894 | Val MAE: 0.3426
Epoch 09 | Train Loss: 0.1835 | Val MAE: 0.3474
Epoch 10 | Train Loss: 0.1815 | Val MAE: 0.2985
Epoch 11 | Train Loss: 0.1797 | Val MAE: 0.3805
Epoch 12 | Train Loss: 0.1781 | Val MAE: 0.3413
Epoch 13 | Train Loss: 0.1766 | Val MAE: 0.4263
Epoch 14 | Train Loss: 0.1751 | Val MAE: 0.3836
Epoch 15 | Train Loss: 0.1717 | Val MAE: 0.2496
Epoch 16 | Train Loss: 0.1720 | Val MAE: 0.2479
Epoch 17 | Train Loss: 0.1715 | Val MAE: 0.3469
Epoch 18 | Train Loss: 0.1709 | Val MAE: 0.3665
Epoch 19 | Train Loss: 0.1683 | Val MAE: 0.3485
Epoch 20 | Train Loss: 0.1664 | Val MAE: 0.3633
✓ Best validation MAE: 0.2479


In [30]:
train_dataloader_unshuffled = load_test_data(
    test_path=TRAIN_PATH,
    batchsize=64,
    pipeline=pipeline,  # Use the same preprocessing as training
    load_anonymized_data=False  # We have ground truth for training data
)

c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_014.csv
c:\mai

In [31]:
import plotly.graph_objects as go

# Set model to evaluation mode
model.eval()

# Storage for predictions and ground truth
total_loss = 0  
preds = []
ground_truths = []

# Generate predictions for all training data
with torch.no_grad():
    for step, (x, y) in enumerate(train_dataloader_unshuffled):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = criterion(pred, y)
        total_loss += loss.item()
        
        # Store predictions and ground truth
        preds.append(pred.reshape(-1).cpu().numpy())
        ground_truths.append(y.reshape(-1).cpu().numpy())

    print(f"Avg. Loss: {total_loss/step}")
    
    # Concatenate all batches into single arrays
    ground_truths = np.concatenate(ground_truths)
    preds = np.concatenate(preds)

# Create interactive plot
fig = go.Figure()
fig.add_trace(go.Scatter(y=ground_truths, mode='lines', name='truth', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=preds.flatten(), mode='lines', name='pred', line=dict(color='red'), opacity=0.8))
fig.update_layout(
    title='Ground Truth vs Predictions',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (degrees)',
    width=1200,
    height=500
)
fig.show()

Avg. Loss: 0.14576483749563118
